# Session 3: Exploratory Data Analysis and Visualisation

**Introduction to Data Analysis with Python**

---

### What this session covers

| Part | Topic | Time |
|------|-------|------|
| A | The plotting landscape | 5 min |
| B | Distributions | 15 min |
| C | Relationships between variables | 15 min |
| D | Correlation heatmap | 10 min |
| E | Saving figures | 5 min |
| F | The CRAFT framework applied | 10 min |

### Learning outcomes

By the end of this session you will be able to:

- Produce histograms, box plots, scatter plots, and bar charts using seaborn
- Customise titles, axis labels, colours, and figure sizes
- Create a correlation heatmap and interpret it
- Save figures to PNG for use in a report or dissertation
- Map your visualisation workflow onto the CRAFT pipeline

### Dataset used in this session

`wdi_2019.csv` — a cross-sectional extract of World Development Indicators for approximately 100 countries (reference year: 2019).

Source: World Bank World Development Indicators. This is a synthetic teaching dataset with the same structure and realistic values.

| Column | Description |
|--------|-------------|
| `country` | Country name |
| `region` | World Bank region |
| `gdp_per_capita` | GDP per capita (USD) |
| `life_expectancy` | Average life expectancy at birth (years) |
| `school_enrollment_pct` | Secondary school enrolment (%) |
| `co2_per_capita` | CO₂ emissions per capita (tonnes) |
| `internet_users_pct` | Internet users (% of population) |
| `fertility_rate` | Total fertility rate (births per woman) |
| `urban_population_pct` | Urban population (% of total) |
| `health_expenditure_pct_gdp` | Health expenditure (% of GDP) |

---

## Part A: The Plotting Landscape

Python has two main plotting libraries:

- **matplotlib** is the foundation — it gives you complete control but requires more code
- **seaborn** builds on matplotlib and produces polished statistical charts with less code

In practice you use both together: seaborn for most charts, matplotlib for fine-tuning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv("wdi_2019.csv")
print(f"Loaded {len(df)} countries across {df['region'].nunique()} regions.")
df.head()

In [ ]:
# Set a clean visual style for all charts in this notebook
sns.set_theme(style="whitegrid", palette="muted")

# Check what regions we have
df["region"].value_counts()

> **How a matplotlib figure is structured:**
>
> ```
> Figure  ← the whole canvas
> └── Axes  ← the plot area (one or more per Figure)
>     ├── x-axis / y-axis
>     ├── title
>     └── plotted data
> ```
>
> Most seaborn functions return an `Axes` object. You customise it with `ax.set_title()`,
> `ax.set_xlabel()`, etc.

---

## Part B: Distributions

### B.1 Histogram

Histograms show how values of a single variable are spread.

```stata
* Stata
histogram gdp_per_capita
```

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

sns.histplot(df["gdp_per_capita"], bins=20, ax=ax)

ax.set_title("Distribution of GDP per Capita (2019)")
ax.set_xlabel("GDP per capita (USD)")
ax.set_ylabel("Number of countries")

plt.tight_layout()
plt.show()

> **What to notice:** the distribution is strongly right-skewed — most countries have low GDP per capita but a few very wealthy countries stretch the tail.
> A log scale often helps here:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Linear scale
sns.histplot(df["gdp_per_capita"], bins=20, ax=axes[0])
axes[0].set_title("Linear scale")
axes[0].set_xlabel("GDP per capita (USD)")

# Log scale
sns.histplot(np.log10(df["gdp_per_capita"]), bins=20, ax=axes[1], color="coral")
axes[1].set_title("Log10 scale")
axes[1].set_xlabel("log10(GDP per capita)")

plt.suptitle("GDP per Capita — Linear vs Log Scale", y=1.02)
plt.tight_layout()
plt.show()

### B.2 Box plot

A box plot shows the median, interquartile range (IQR), and outliers at a glance — very useful for comparing groups.

```stata
* Stata
graph box life_expectancy, over(region)
```

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Order regions by median life expectancy
order = (df.groupby("region")["life_expectancy"]
           .median()
           .sort_values(ascending=False)
           .index)

sns.boxplot(data=df, x="region", y="life_expectancy", order=order, ax=ax)

ax.set_title("Life Expectancy by World Bank Region (2019)")
ax.set_xlabel("")
ax.set_ylabel("Life expectancy (years)")
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

> **Reading the box:** the line in the middle is the median. The box spans the IQR (25th to 75th percentile). The whiskers extend to 1.5 × IQR. Points beyond that are outliers.

---

## Part C: Relationships Between Variables

### C.1 Scatter plot

Scatter plots show how two continuous variables relate to each other.

```stata
* Stata
scatter life_expectancy gdp_per_capita
```

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="gdp_per_capita",
    y="life_expectancy",
    hue="region",      # colour points by region
    s=60,              # point size
    alpha=0.8,
    ax=ax
)

ax.set_title("GDP per Capita vs Life Expectancy (2019)")
ax.set_xlabel("GDP per capita (USD)")
ax.set_ylabel("Life expectancy (years)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

> **What to notice:** as GDP rises, life expectancy rises too — but the relationship is not linear. The biggest gains come at lower income levels. This is a classic pattern in development economics.

### C.2 Scatter plot with a regression line

```stata
* Stata
graph twoway (scatter life_expectancy gdp_per_capita) (lfit life_expectancy gdp_per_capita)
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Raw GDP — linear scale
sns.regplot(data=df, x="gdp_per_capita", y="life_expectancy",
            scatter_kws={"alpha": 0.5}, ax=axes[0])
axes[0].set_title("Linear scale")
axes[0].set_xlabel("GDP per capita (USD)")
axes[0].set_ylabel("Life expectancy (years)")

# Log GDP — better fit
df["log_gdp"] = np.log10(df["gdp_per_capita"])
sns.regplot(data=df, x="log_gdp", y="life_expectancy",
            scatter_kws={"alpha": 0.5}, color="coral", ax=axes[1])
axes[1].set_title("Log scale (better fit)")
axes[1].set_xlabel("log10(GDP per capita)")
axes[1].set_ylabel("Life expectancy (years)")

plt.suptitle("Regression Line: GDP per Capita vs Life Expectancy")
plt.tight_layout()
plt.show()

### C.3 Bar chart — comparing means across groups

```stata
* Stata
graph bar internet_users_pct, over(region)
```

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

order = (df.groupby("region")["internet_users_pct"]
           .mean()
           .sort_values(ascending=False)
           .index)

sns.barplot(data=df, x="region", y="internet_users_pct",
            order=order, errorbar="sd", ax=ax)

ax.set_title("Internet Users by Region — Mean and Standard Deviation (2019)")
ax.set_xlabel("")
ax.set_ylabel("Internet users (% of population)")
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

---

## Part D: Correlation Heatmap

A correlation heatmap shows pairwise correlations between all numeric variables at once. It is a quick way to spot which variables are related before diving into more detailed analysis.

In [ ]:
# Compute the correlation matrix — only for numeric columns
numeric_cols = ["gdp_per_capita", "life_expectancy", "school_enrollment_pct",
                "co2_per_capita", "internet_users_pct",
                "fertility_rate", "urban_population_pct",
                "health_expenditure_pct_gdp"]

corr = df[numeric_cols].corr().round(2)
corr

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

sns.heatmap(
    corr,
    annot=True,          # show values in each cell
    fmt=".2f",           # two decimal places
    cmap="coolwarm",     # red = positive, blue = negative
    center=0,
    vmin=-1, vmax=1,
    square=True,
    ax=ax
)

ax.set_title("Correlation Matrix — World Development Indicators (2019)")
plt.tight_layout()
plt.show()

> **Reading the heatmap:**
> - Values close to **+1** (dark red) = strong positive correlation: as one variable rises, so does the other
> - Values close to **-1** (dark blue) = strong negative correlation: as one rises, the other falls
> - Values close to **0** (white) = little linear relationship
>
> **What stands out?**
> - `fertility_rate` is strongly *negatively* correlated with `life_expectancy`, `gdp_per_capita`, and `internet_users_pct`
> - `life_expectancy`, `internet_users_pct`, and `school_enrollment_pct` are all strongly *positively* correlated with each other

---

## Part E: Saving Figures

Once you have a chart you're happy with, save it to file for use in a report or dissertation.

```stata
* Stata
graph export fig.png, replace
```

In [ ]:
# Reproduce the scatter plot and save it
fig, ax = plt.subplots(figsize=(9, 6))

sns.scatterplot(
    data=df, x="log_gdp", y="life_expectancy",
    hue="region", s=60, alpha=0.8, ax=ax
)
ax.set_title("GDP per Capita vs Life Expectancy by Region (2019)")
ax.set_xlabel("GDP per capita (log scale, USD)")
ax.set_ylabel("Life expectancy (years)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

plt.tight_layout()

# Save — dpi=150 is fine for screen; use dpi=300 for print
fig.savefig("wdi_gdp_vs_life_expectancy.png", dpi=150)
print("Figure saved.")
plt.show()

---

## Part F: The CRAFT Framework Applied

CRAFT provides a discipline for any data workflow. Here is how this session maps onto it:

| Step | What it means | What we did in this session |
|------|--------------|-----------------------------|
| **Collect** | Bring your data together | Loaded `wdi_2019.csv` with `pd.read_csv()` |
| **Refine** | Clean and standardise | Checked for missing values; added `log_gdp` column |
| **Analyse** | Ask questions of the data | Computed correlations; compared regions; spotted the GDP/life expectancy relationship |
| **Fact-check** | Verify before publishing | The cell below demonstrates this step |
| **Transform** | Produce the output you need | Saved a publication-ready PNG with `fig.savefig()` |

The Fact-check step is the one most often skipped under time pressure — but it is the most important one.

### F.1 Fact-check: verify a headline figure

Before quoting any number from your analysis, trace it back to the underlying data.

In [ ]:
# Claim: 'North America has the highest mean internet usage.'
# Let's verify this is actually what the data shows.

region_internet = (df.groupby("region")["internet_users_pct"]
                     .mean()
                     .round(1)
                     .sort_values(ascending=False))

print("Mean internet users (%) by region:")
print(region_internet)
print()

top_region = region_internet.index[0]
top_value  = region_internet.iloc[0]
print(f"Verified: '{top_region}' has the highest mean internet usage: {top_value}%")
print()

# How many countries is that mean based on?
n_countries = df[df["region"] == top_region].shape[0]
print(f"Based on {n_countries} countries — always report the sample size.")

> **The CRAFT fact-check habit in practice:**
>
> 1. Before quoting a number, re-derive it from the data with a fresh piece of code
> 2. Check how many observations the number is based on — a mean from 2 countries means less than one from 20
> 3. If the number looks surprising, look at the underlying rows rather than the aggregate
>
> This takes two minutes and can prevent serious errors in a report or presentation.

---

## Exercises

---

### Exercise 1 — Fertility rate distribution

Create a histogram of `fertility_rate`. Then create a box plot showing fertility rate broken down by region. What pattern do you see?

In [ ]:
# Exercise 1 — your code here

---

### Exercise 2 — Scatter plot of your choosing

Pick any two numeric columns from the dataset and create a scatter plot, colouring points by region. Add a title and axis labels. What relationship do you observe?

In [ ]:
# Exercise 2 — your code here

---

### Exercise 3 — CRAFT fact-check

From the correlation heatmap, `fertility_rate` and `life_expectancy` appear strongly negatively correlated.

1. Compute the exact correlation coefficient between these two variables
2. Find the country with the highest fertility rate and the country with the lowest — does this make sense?
3. Write one sentence you could include in a report, citing the specific value

*(Hint: `df.nlargest(1, "fertility_rate")` gives the row with the highest value.)*

In [ ]:
# Exercise 3 — your code here

---

### Exercise 4 — Save a chart (stretch goal)

Recreate your scatter plot from Exercise 2 with polished formatting and save it as `my_scatter.png` at 300 dpi.

In [ ]:
# Exercise 4 — your code here

---

## Wrap-up

### What you covered today

- matplotlib and seaborn as a two-layer plotting system
- Histograms and log-scale transformations for skewed data
- Box plots for comparing distributions across groups
- Scatter plots with colour encoding for a third variable
- Adding regression lines with `sns.regplot()`
- Correlation heatmaps for a bird's-eye view of all pairwise relationships
- Saving figures to file with `fig.savefig()`
- Applying the CRAFT framework: Analyse, Fact-check, Transform

### Coming up in Session 4

Session 4 introduces formal **statistical analysis**: t-tests, chi-square tests, and OLS regression using `scipy` and `statsmodels`. You will work with the Health Survey for England dataset and produce results tables you could include in a dissertation.

### Save your work

**File → Save Notebook** (or Cmd+S).